<a href="https://colab.research.google.com/github/Aash77-b/new.md/blob/main/Copy_of_Digit_Recognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ipycanvas installation

The `!pip install ipycanvas` command installs the `ipycanvas` library. This library provides a Jupyter widget that allows for drawing on an HTML5 canvas directly within Jupyter notebooks, including Google Colab.

In [ ]:
!pip install ipycanvas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.0/143.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.4 MB/s eta 0:00:00



I imported various libraries that are essential for this project:

*   **`torch`, `torch.nn`, `torch.optim`**: These are core PyTorch libraries used for building, training, and optimizing neural networks. `torch` provides fundamental tensor operations, `torch.nn` defines neural network modules, and `torch.optim` contains optimization algorithms.
*   **`torchvision`, `torchvision.transforms`**: `torchvision` is a PyTorch domain library for computer vision tasks. `torchvision.transforms` provides common image transformations to prepare data for neural networks.
*   **`matplotlib.pyplot`, `numpy`**: These are standard Python libraries for numerical operations (`numpy`) and creating static, animated, and interactive visualizations (`matplotlib.pyplot`).
*   **`ipycanvas`**: This library enables drawing directly onto a canvas widget within a Jupyter/Colab notebook, which is used here for the interactive digit drawing.
*   **`IPython.display`**: This module provides tools for displaying rich content in IPython environments, such as widgets or images.
*   **`PIL.Image` (Pillow)**: This is the Python Imaging Library, used for opening, manipulating, and saving many different image file formats.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

from ipycanvas import Canvas
from IPython.display import display
from PIL import Image

from google.colab import output
output.enable_custom_widget_manager()

## Data Loading and Preparation

the loading and initial preparation of the MNIST dataset, which consists of handwritten digits.

1.  **Transformation**: `transforms.ToTensor()` is used to convert images loaded from the dataset into PyTorch tensors. This also scales the pixel values from the original 0-255 range to a 0.0-1.0 range, which is standard practice for neural network inputs.

2.  **Dataset Loading**:
    *   `torchvision.datasets.MNIST` is used to download (if not already present) and load the MNIST dataset.
    *   `train_dataset` is created for the training data (`train=True`).
    *   `test_dataset` is created for the testing data (`train=False`).

3.  **DataLoaders**:
    *   `torch.utils.data.DataLoader` wraps the datasets to provide an iterable over the dataset, supporting batching, shuffling, and multi-process data loading.
    *   `train_loader` is configured with `batch_size=64` and `shuffle=True` to randomize the order of training samples in each epoch, which helps in better model generalization.
    *   `test_loader` is also configured with `batch_size=64` but `shuffle` is set to `False` as shuffling is not necessary for evaluation.

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

100%|██████████| 9.91M/9.91M [00:00<00:00, 35.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 904kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.29MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.3MB/s]


## Neural Network Architecture Explanation

The `NeuralNet` class defines our neural network model. It inherits from `nn.Module`, which is the base class for all neural network modules in PyTorch. This inheritance provides the fundamental functionalities for building and training neural networks.

The network is composed of a sequential stack of layers, defined within the `nn.Sequential` container:

1.  **Input Layer to First Hidden Layer**: `nn.Linear(784, 128)`
    *   This is a fully connected layer that takes an input of 784 features (representing the 28x28 pixel MNIST images flattened into a single vector) and transforms it into an output of 128 features.

2.  **Activation Function**: `nn.ReLU()`
    *   A Rectified Linear Unit (ReLU) activation function is applied to the output of the first hidden layer. It introduces non-linearity, allowing the model to learn more complex patterns.

3.  **First Hidden Layer to Second Hidden Layer**: `nn.Linear(128, 64)`
    *   Another fully connected layer that takes the 128 features from the previous layer and reduces them to 64 features.

4.  **Activation Function**: `nn.ReLU()`
    *   Another ReLU activation function is applied to the output of the second hidden layer.

5.  **Second Hidden Layer to Output Layer**: `nn.Linear(64, 10)`
    *   The final fully connected layer takes the 64 features and maps them to 10 output features, corresponding to the 10 possible digit classes (0-9).

### Forward Pass

The `forward(self, x)` method defines how input data `x` is processed through the network. When data is passed to an instance of `NeuralNet` (e.g., `model(images)`), this method is called. It simply passes the input `x` through the `self.model` sequential container, which applies all the defined layers in order, from the input layer to the final output layer.

In [ ]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(784,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        return self.model(x)

### Model Instantiation

Before training our neural network, we need to create an instance of the `NeuralNet` class. The line `model = NeuralNet()` does exactly this: it creates an object named `model` which is an instance of our `NeuralNet` class. This `model` object now holds the neural network architecture we defined (the sequential layers), making it ready for the next steps of training and prediction.

In [ ]:
model = NeuralNet()

## Loss Function and Optimizer

To train our neural network, we need to define a **loss function** and an **optimizer**.

### Loss Function: `nn.CrossEntropyLoss()`
`nn.CrossEntropyLoss()` is a common loss function used for multi-class classification problems, such as digit recognition. It combines `nn.LogSoftmax()` and `nn.NLLLoss()` in one single class. It is particularly suitable for problems where the output classes are mutually exclusive (an image can only be one digit).

It computes the negative log likelihood loss. The goal during training is to minimize this loss, which means making the predicted probabilities for the correct class as high as possible.

### Optimizer: `optim.Adam()`
`optim.Adam()` is an optimization algorithm used to update the weights and biases of the neural network during training. Adam (Adaptive Moment Estimation) is an adaptive learning rate optimization algorithm that has been widely adopted in deep learning due to its efficiency and good performance in practice. It combines the advantages of two other extensions of stochastic gradient descent: Adaptive Gradient Algorithm (AdaGrad) and Root Mean Square Propagation (RMSProp).

Key features of Adam:
*   **Adaptive Learning Rates**: It computes individual adaptive learning rates for different parameters from estimates of first and second moments of the gradients.
*   **Efficiency**: It's computationally efficient and requires little memory.
*   **Robustness**: It generally works well in practice and is robust to the choice of hyperparameters.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Training Time! 🚀

next cell is where the magic happens,we're training our neural network! Here's a quick rundown:

*   **`epochs = 8`**: This means our model will go through the entire training dataset 8 times. Each 'epoch' is like a full study session for the network.
*   **Model, Criterion, Optimizer Re-instantiation**: We're just making sure we have a fresh start with our `NeuralNet` model, `CrossEntropyLoss` (our error calculator), and `Adam` optimizer (our learning guide).
*   **The Loop**: We iterate through each `epoch`. Inside that, we go through all the `images` and `labels` (the correct digits) in our `train_loader` in batches.
    *   `images = images.view(-1, 784)`: We're flattening each 28x28 pixel image into a single line of 784 numbers so our `Linear` layers can understand them.
    *   `outputs = model(images)`: We feed these flattened images to our model to get its predictions.
    *   `loss = criterion(outputs, labels)`: We compare the model's predictions with the actual `labels` to calculate how wrong it was. The smaller the `loss`, the better!
    *   `optimizer.zero_grad()`: This clears any old 'error memory' from the previous step.
    *   `loss.backward()`: This is the crucial step where the model figures out how to adjust its internal workings (weights) to reduce the error.
    *   `optimizer.step()`: The optimizer then applies those adjustments.
    *   `total_loss += loss.item()`: We keep track of the total error for the epoch.
*   **`print(...)`**: After each epoch, we print out the average loss. You'll notice it goes down over time, meaning our model is learning and getting better at recognizing digits!

In [ ]:
epochs = 15

model = NeuralNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):

    total_loss = 0

    for images, labels in train_loader:

        images = images.view(-1, 784)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

Epoch: 1 Loss: 0.28116835757637265
Epoch: 2 Loss: 0.10484773669638105
Epoch: 3 Loss: 0.07065789354369362
Epoch: 4 Loss: 0.05211586058551847
Epoch: 5 Loss: 0.039747357110009846
Epoch: 6 Loss: 0.031100889043222224
Epoch: 7 Loss: 0.02415063736940993
Epoch: 8 Loss: 0.022794044469198985
Epoch: 9 Loss: 0.018562488785973674
Epoch: 10 Loss: 0.016042250449411793
Epoch: 11 Loss: 0.014002003192357296
Epoch: 12 Loss: 0.013032211232091593
Epoch: 13 Loss: 0.012004112474205846
Epoch: 14 Loss: 0.011053965976020519
Epoch: 15 Loss: 0.009892900706716697


### so now i set my canvas for my amalye handwriting😉!



*   **`canvas = Canvas(width=280, height=280)`**: We're creating a canvas that's 280 pixels by 280 pixels. This size is good because we'll eventually shrink your drawing down to 28x28 for our model.
*   **`canvas.stroke_style = "white"`**: When you draw a line, it'll be white.
*   **`canvas.line_width = 15`**: Your drawing 'pen' will be quite thick, making it easy to draw clear digits.
*   **`canvas.fill_style = "black"`**: The background of our canvas is set to black, providing good contrast for your white drawings.
*   **`canvas.fill_rect(0,0,280,280)`**: This line actually fills the entire canvas with black, so it's ready for you to draw.
*   **`canvas.sync_image_data = True`**: This is important! It ensures that when you draw, the canvas actively updates its internal image data, which we'll need later to grab your drawing.
*   **`display(canvas)`**: Finally, this shows the canvas right here in your notebook, ready for your artistic endeavors! Go ahead and draw a digit after the next cells run.

In [ ]:
from ipycanvas import hold_canvas

canvas = Canvas(width=280, height=280)

canvas.stroke_style = "white"
canvas.line_width = 15
canvas.fill_style = "black"
canvas.fill_rect(0,0,280,280)

canvas.sync_image_data = True

display(canvas)

Canvas(height=280, sync_image_data=True, width=280)

### Making the Canvas Interactive! 🖱️

This cell gives our canvas the ability to respond to your mouse movements, letting you draw.

*   **`drawing = False`, `last_x`, `last_y`**: These are helper variables. `drawing` tells us if you're currently holding down the mouse button. `last_x` and `last_y` remember where your mouse was last, so we can draw continuous lines.
*   **`on_mouse_down(x, y)`**: When you *press* the mouse button, this function says, "Okay, start drawing from here!" It sets `drawing` to `True` and records your starting `x` and `y` coordinates.
*   **`on_mouse_move(x, y)`**: If you're *holding down* the mouse button and *moving* it (i.e., `drawing` is `True`):
    *   `with hold_canvas(canvas)`: This makes drawing smoother, updating the canvas efficiently.
    *   `canvas.begin_path()`, `canvas.move_to(last_x, last_y)`, `canvas.line_to(x, y)`, `canvas.stroke()`: These lines draw a white line from your `last_x, last_y` to your current `x, y` position, creating the stroke of your digit.
    *   `last_x = x`, `last_y = y`: We update your `last_x` and `last_y` to your current position, so the next line segment starts exactly where this one ended.
*   **`on_mouse_up(x, y)`**: When you *release* the mouse button, this function simply says, "Okay, stop drawing." It sets `drawing` back to `False`.
*   **`canvas.on_mouse_down(...)`, etc.**: These lines connect our functions to the actual mouse events on the canvas, so they know when to run. Pretty neat, huh?

In [ ]:
drawing = False
last_x = 0
last_y = 0

def on_mouse_down(x, y):
    global drawing, last_x, last_y
    drawing = True
    last_x = x
    last_y = y

def on_mouse_move(x, y):
    global drawing, last_x, last_y
    if drawing:
        with hold_canvas(canvas):
            canvas.begin_path()
            canvas.move_to(last_x, last_y)
            canvas.line_to(x, y)
            canvas.stroke()
        last_x = x
        last_y = y

def on_mouse_up(x, y):
    global drawing
    drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

Support for third party widgets will remain active for the duration of the session. To disable support:



This `get_image()` function is super important because it takes your beautiful handwriting(dont meshkormem now) from the canvas and gets it ready for our neural network.

*   **`image = np.array(canvas.get_image_data())`**: This line grabs the pixel data directly from your canvas as a NumPy array. It's essentially a snapshot of what you've drawn.
*   **`image = Image.fromarray(image[:,:,0])`**: The `ipycanvas` gives us RGB data (Red, Green, Blue, and Alpha channels), but since we only drew in white on black, all color channels are the same. We just need one channel (the `0` index) to represent the grayscale image, and `Pillow` helps us create an image object from it.
*   **`image = image.resize((28,28))`**: This is crucial! Our neural network was trained on 28x28 pixel images, so we need to shrink your drawing down to that size. This is why a thicker line width helps – it makes your drawing more robust to resizing.
*   **`image = np.array(image)`**: We convert the resized Pillow image back into a NumPy array.
*   **`image = image / 255.0`**: Image pixel values usually range from 0 to 255. Our neural network expects values between 0.0 and 1.0, so we normalize them by dividing by 255.
*   **`image = torch.tensor(image).float()`**: We convert the NumPy array into a PyTorch tensor, which is the format our model expects.
*   **`image = image.view(1,784)`**: Remember how we flattened the MNIST images to 784 pixels? We do the same here. The `1` means we're processing a single image. Now, your drawing is ready for prediction!

In [ ]:
def get_image():

    image = np.array(canvas.get_image_data())

    image = Image.fromarray(image[:,:,0])

    image = image.resize((28,28))

    image = np.array(image)

    image = image / 255.0

    image = torch.tensor(image).float()

    image = image.view(1,784)

    return image

### What Digit u draw?! 🧐

The `predict()` function is where our trained model tries to guess what digit you just drew!

*   **`image = get_image()`**: First, it calls our `get_image()` function to process your canvas drawing into the right format for the model.
*   **`output = model(image)`**: This is it! We feed your prepared image to our `model`. The model then outputs a set of 10 numbers, each representing how confident it is that your drawing is a particular digit (0 through 9).
*   **`_, prediction = torch.max(output,1)`**: The `torch.max()` function looks at those 10 confidence scores and finds the highest one. The `prediction` variable then stores the *index* of that highest score, which corresponds to the digit the model thinks you drew.
*   **`print("Predicted Digit:", prediction.item())`**: Finally, we print out the model's prediction! Go ahead and run this cell after drawing something on the canvas to see what it thinks!

In [ ]:
def predict():

    image = get_image()

    output = model(image)

    _, prediction = torch.max(output,1)

    print("Predicted Digit:", prediction.item())

In [ ]:
predict()

Predicted Digit: 7


### Fresh Canvas!

The `clear_canvas()` function does exactly what it says – it wipes your canvas clean so you can draw a new digit!

*   **`canvas.fill_style = "black"`**: It sets the fill color back to black.
*   **`canvas.fill_rect(0,0,280,280)`**: Then it fills the entire canvas with black, effectively erasing your previous drawing.

Run this cell whenever you want to start a new drawing!

In [ ]:
def clear_canvas():
    canvas.fill_style = "black"
    canvas.fill_rect(0,0,280,280)

clear_canvas()